# Practical Exercise: RAG with Knowledge Graphs
### Big Data – Lecture Series | KIT / Hector School

---

## Overview

In this exercise, you will build a **Retrieval-Augmented Generation (RAG)** pipeline that leverages a **Knowledge Graph** as its retrieval backend. 

Standard RAG systems retrieve chunks of text from a vector store. Here, we go one step further: we store facts as **RDF triples** (subject–predicate–object), query the graph using **SPARQL**, and then use the retrieved facts to ground the answers of a language model.

### Learning Objectives
By the end of this exercise you will be able to:
1. Build a small **RDF knowledge graph** using `rdflib`
2. Query it with **SPARQL**
3. Implement a simple **TF-IDF vector retrieval** baseline over text documents
4. Combine graph facts + retrieved text into a **RAG prompt**
5. Compare *graph-augmented* vs *vector-only* retrieval

### Tools & Libraries
| Library | Purpose |
|---|---|
| `rdflib` | Build and query RDF knowledge graphs |
| `sklearn` | TF-IDF vectorizer for vector retrieval |
| `networkx` + `matplotlib` | Visualize the knowledge graph |
| `google-genai` (optional) | Call Gemini as the LLM backend |

---

## Part 0 – Setup

In [ ]:
# Install required libraries (run once)
!pip install rdflib scikit-learn networkx matplotlib google-genai --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# Core libraries
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, OWL, XSD
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

print('✅ All imports successful!')

---
## Part 1 – Build a Knowledge Graph about Big Data Technologies

We model a small domain about **Big Data technologies** using RDF triples.  
Recall from the lectures: a triple has the form `(subject, predicate, object)`.  
For example: `(Hadoop, rdf:type, Framework)` or `(Hadoop, uses, HDFS)`.

In [ ]:
# ── Namespaces ──────────────────────────────────────────────────────────────
BD  = Namespace("http://bigdata.example.org/")   # our domain
DBR = Namespace("http://dbpedia.org/resource/")  # DBpedia resources (as in lecture)
DBO = Namespace("http://dbpedia.org/ontology/")  # DBpedia ontology

# ── Create the RDF Graph ─────────────────────────────────────────────────────
g = Graph()
g.bind("bd",  BD)
g.bind("dbr", DBR)
g.bind("dbo", DBO)

# ── Helper to add triples more concisely ─────────────────────────────────────
def add(s, p, o):
    g.add((s, p, o))

# ── Classes (ontology layer) ──────────────────────────────────────────────────
add(BD.Framework,     RDF.type,          RDFS.Class)
add(BD.StorageSystem, RDF.type,          RDFS.Class)
add(BD.QueryLanguage, RDF.type,          RDFS.Class)
add(BD.Database,      RDF.type,          RDFS.Class)
add(BD.Database,      RDFS.subClassOf,   BD.StorageSystem)

# ── Instances ─────────────────────────────────────────────────────────────────
# Hadoop
add(BD.Hadoop,   RDF.type,          BD.Framework)
add(BD.Hadoop,   RDFS.label,        Literal("Apache Hadoop", lang="en"))
add(BD.Hadoop,   BD.uses,           BD.HDFS)
add(BD.Hadoop,   BD.uses,           BD.MapReduce)
add(BD.Hadoop,   BD.developedBy,    BD.Apache)
add(BD.Hadoop,   BD.processingType, Literal("batch"))

# Spark
add(BD.Spark,    RDF.type,          BD.Framework)
add(BD.Spark,    RDFS.label,        Literal("Apache Spark", lang="en"))
add(BD.Spark,    BD.uses,           BD.RDD)
add(BD.Spark,    BD.developedBy,    BD.Apache)
add(BD.Spark,    BD.processingType, Literal("batch"))
add(BD.Spark,    BD.processingType, Literal("streaming"))
add(BD.Spark,    BD.fasterThan,     BD.Hadoop)

# HDFS
add(BD.HDFS,     RDF.type,          BD.StorageSystem)
add(BD.HDFS,     RDFS.label,        Literal("Hadoop Distributed File System", lang="en"))
add(BD.HDFS,     BD.replicationFactor, Literal(3, datatype=XSD.integer))

# Kafka
add(BD.Kafka,    RDF.type,          BD.Framework)
add(BD.Kafka,    RDFS.label,        Literal("Apache Kafka", lang="en"))
add(BD.Kafka,    BD.developedBy,    BD.Apache)
add(BD.Kafka,    BD.processingType, Literal("streaming"))
add(BD.Kafka,    BD.usedWith,       BD.Spark)

# NoSQL Databases
add(BD.Cassandra, RDF.type,         BD.Database)
add(BD.Cassandra, RDFS.label,       Literal("Apache Cassandra", lang="en"))
add(BD.Cassandra, BD.developedBy,   BD.Apache)
add(BD.Cassandra, BD.dataModel,     Literal("wide-column"))
add(BD.Cassandra, BD.CAP,           Literal("AP"))

add(BD.MongoDB,  RDF.type,          BD.Database)
add(BD.MongoDB,  RDFS.label,        Literal("MongoDB", lang="en"))
add(BD.MongoDB,  BD.dataModel,      Literal("document"))
add(BD.MongoDB,  BD.CAP,            Literal("CP"))

add(BD.Neo4j,    RDF.type,          BD.Database)
add(BD.Neo4j,    RDFS.label,        Literal("Neo4j", lang="en"))
add(BD.Neo4j,    BD.dataModel,      Literal("graph"))
add(BD.Neo4j,    BD.queryLanguage,  BD.Cypher)

# Query Languages
add(BD.SPARQL,   RDF.type,          BD.QueryLanguage)
add(BD.SPARQL,   RDFS.label,        Literal("SPARQL", lang="en"))
add(BD.SPARQL,   BD.usedFor,        BD.RDFGraph)

add(BD.Cypher,   RDF.type,          BD.QueryLanguage)
add(BD.Cypher,   RDFS.label,        Literal("Cypher", lang="en"))

add(BD.HiveQL,   RDF.type,          BD.QueryLanguage)
add(BD.HiveQL,   BD.usedFor,        BD.Hadoop)

# AWS
add(BD.S3,       RDF.type,          BD.StorageSystem)
add(BD.S3,       RDFS.label,        Literal("Amazon S3", lang="en"))
add(BD.S3,       BD.providedBy,     BD.AWS)
add(BD.EMR,      BD.runs,           BD.Spark)
add(BD.EMR,      BD.runs,           BD.Hadoop)

print(f'✅ Knowledge Graph built: {len(g)} triples')

### 1.1 Visualize the Knowledge Graph

In [ ]:
def short(uri):
    """Return the local name of a URI for display."""
    if isinstance(uri, Literal):
        return str(uri)
    return str(uri).split('/')[-1].split('#')[-1]

# Build a NetworkX graph for visualization
G_vis = nx.DiGraph()

# Only include URIRef → URIRef edges (skip Literals and rdf:type schema triples)
skip_predicates = {RDF.type, RDFS.subClassOf, RDFS.label}
node_types = {}

for s, p, o in g:
    if p == RDF.type and not isinstance(o, Literal):
        node_types[short(s)] = short(o)  # track class membership
    if isinstance(s, URIRef) and isinstance(o, URIRef) and p not in skip_predicates:
        G_vis.add_edge(short(s), short(o), label=short(p))

# Color nodes by type
color_map = {
    'Framework':     '#4C9BE8',
    'Database':      '#F4A261',
    'StorageSystem': '#2A9D8F',
    'QueryLanguage': '#E76F51',
}
default_color = '#A8DADC'

node_colors = [color_map.get(node_types.get(n, ''), default_color) for n in G_vis.nodes()]

fig, ax = plt.subplots(figsize=(14, 9))
pos = nx.spring_layout(G_vis, seed=42, k=2.2)
nx.draw_networkx_nodes(G_vis, pos, node_color=node_colors, node_size=1600, ax=ax, alpha=0.9)
nx.draw_networkx_labels(G_vis, pos, font_size=8, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G_vis, pos, edge_color='#555', arrows=True,
                       arrowsize=15, width=1.2, ax=ax, connectionstyle='arc3,rad=0.1')
edge_labels = nx.get_edge_attributes(G_vis, 'label')
nx.draw_networkx_edge_labels(G_vis, pos, edge_labels=edge_labels, font_size=6.5,
                             font_color='#333', ax=ax)

# Legend
legend = [mpatches.Patch(color=c, label=l) for l, c in color_map.items()]
ax.legend(handles=legend, loc='upper left', fontsize=9)
ax.set_title('Big Data Technologies – Knowledge Graph', fontsize=14, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

---
## Part 2 – Query the Knowledge Graph with SPARQL

SPARQL is the standard query language for RDF graphs (analogous to SQL for relational databases).  
A SPARQL `SELECT` query uses **triple patterns** with variables (prefixed with `?`).

In [ ]:
def run_sparql(query_str: str, title: str = ''):
    """Run a SPARQL query and print results as a table."""
    results = g.query(query_str)
    if title:
        print(f'\n📊 {title}')
        print('─' * 50)
    vars_ = [str(v) for v in results.vars]
    print('  |  '.join(vars_))
    print('─' * 50)
    for row in results:
        print('  |  '.join(short(v) if v else '–' for v in row))
    return list(results)

In [ ]:
# ── Q1: All Frameworks ──────────────────────────────────────────────────────
q1 = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?framework ?label WHERE {
    ?framework a bd:Framework .
    OPTIONAL { ?framework rdfs:label ?label . FILTER(lang(?label) = 'en') }
}
"""
run_sparql(q1, "Q1: All Frameworks")

In [ ]:
# ── Q2: Technologies developed by Apache ───────────────────────────────────
q2 = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?tech ?label WHERE {
    ?tech bd:developedBy bd:Apache .
    OPTIONAL { ?tech rdfs:label ?label . FILTER(lang(?label) = 'en') }
}
"""
run_sparql(q2, "Q2: Technologies developed by Apache")

In [ ]:
# ── Q3: Databases and their data models ────────────────────────────────────
q3 = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?db ?model ?cap WHERE {
    ?db a bd:Database .
    OPTIONAL { ?db bd:dataModel ?model }
    OPTIONAL { ?db bd:CAP ?cap }
}
"""
run_sparql(q3, "Q3: Databases, data models, and CAP properties")

In [ ]:
# ── Q4: Streaming-capable technologies ─────────────────────────────────────
q4 = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?tech ?label WHERE {
    ?tech bd:processingType "streaming" .
    OPTIONAL { ?tech rdfs:label ?label . FILTER(lang(?label) = 'en') }
}
"""
run_sparql(q4, "Q4: Technologies supporting streaming processing")

### ✏️ Exercise 2.1 – Write your own SPARQL queries

Try to answer the following questions by writing SPARQL queries:

1. Which technologies does Spark support (via `bd:uses` or `bd:usedWith`)?
2. Which graph database is available and what query language does it use?
3. Which technologies run on AWS EMR?

In [ ]:
# Your SPARQL queries here:

# Exercise 2.1.1 – Technologies related to Spark
q_ex1 = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?tech WHERE {
    # TODO: write a pattern that finds all technologies Spark uses or is used with
}
"""
# run_sparql(q_ex1, "Exercise 2.1.1")

---
## Part 3 – Graph-Based Retrieval for RAG

In a standard RAG system, a user query is matched against a corpus of text chunks.  
Here, we use the **Knowledge Graph as the retrieval source**: given a natural language question,  
we identify relevant entities and **retrieve their subgraph** via SPARQL.

In [ ]:
# ── Entity Linker: map question keywords → graph entities ──────────────────

# A simple keyword → URI lookup table (in production: use NER + entity linking)
ENTITY_MAP = {
    "hadoop":    BD.Hadoop,
    "spark":     BD.Spark,
    "kafka":     BD.Kafka,
    "cassandra": BD.Cassandra,
    "mongodb":   BD.MongoDB,
    "mongo":     BD.MongoDB,
    "neo4j":     BD.Neo4j,
    "hdfs":      BD.HDFS,
    "sparql":    BD.SPARQL,
    "cypher":    BD.Cypher,
    "s3":        BD.S3,
    "emr":       BD.EMR,
    "aws":       BD.AWS,
    "apache":    BD.Apache,
}

def link_entities(question: str) -> list:
    """Return URIs for entities mentioned in the question."""
    words = question.lower().split()
    found = []
    for word in words:
        # strip punctuation
        word = word.strip('.,?!;:')
        if word in ENTITY_MAP:
            found.append(ENTITY_MAP[word])
    return list(set(found))

# Test
q_test = "What processing types does Spark support?"
entities = link_entities(q_test)
print(f'Entities found for "{q_test}":')
for e in entities:
    print(f'  {e}')

In [ ]:
def retrieve_subgraph_facts(entity_uri: URIRef) -> list[str]:
    """
    Retrieve all facts about an entity from the KG.
    Returns a list of natural-language fact strings.
    """
    facts = []
    # Outgoing triples: entity is the subject
    for s, p, o in g.triples((entity_uri, None, None)):
        pred = short(p)
        obj  = short(o)
        subj = short(s)
        if pred not in ('type', 'label'):  # skip schema triples
            facts.append(f"{subj} {pred} {obj}")
    # Incoming triples: entity is the object
    for s, p, o in g.triples((None, None, entity_uri)):
        pred = short(p)
        obj  = short(o)
        subj = short(s)
        if pred not in ('type', 'label', 'subClassOf'):
            facts.append(f"{subj} {pred} {obj}")
    return facts


def kg_retrieve(question: str) -> str:
    """
    Full KG retrieval pipeline:
    1. Link entities in the question
    2. Retrieve their facts from the graph
    3. Return as a formatted context string
    """
    entities = link_entities(question)
    if not entities:
        return "[No relevant entities found in the knowledge graph]"
    
    all_facts = []
    for entity in entities:
        facts = retrieve_subgraph_facts(entity)
        all_facts.extend(facts)
    
    # Deduplicate
    all_facts = list(dict.fromkeys(all_facts))
    
    context = "\n".join(f"  - {f}" for f in all_facts)
    return context


# Test retrieval
question = "What processing types does Spark support and what does it use?"
print(f'Question: {question}')
print('\nKG-Retrieved Facts:')
print(kg_retrieve(question))

---
## Part 4 – Vector Retrieval (TF-IDF Baseline)

We also implement a classic **vector-based retrieval** using TF-IDF.  
This will serve as a baseline to compare against graph-based retrieval.

In [ ]:
# ── Corpus: text documents about Big Data technologies ─────────────────────
documents = [
    {
        'id': 'doc_hadoop',
        'text': (
            "Apache Hadoop is an open-source framework for distributed storage and batch processing "
            "of large datasets. It uses HDFS (Hadoop Distributed File System) for storage and the "
            "MapReduce programming model for computation. Hadoop was developed by Apache and is "
            "widely used for large-scale data processing. HDFS replicates data blocks three times "
            "by default for fault tolerance."
        )
    },
    {
        'id': 'doc_spark',
        'text': (
            "Apache Spark is a fast, general-purpose cluster computing system developed by Apache. "
            "It supports both batch and streaming processing, making it more versatile than Hadoop "
            "for many workloads. Spark is significantly faster than Hadoop due to in-memory "
            "computation using Resilient Distributed Datasets (RDDs). It integrates well with "
            "Kafka for real-time data streaming pipelines."
        )
    },
    {
        'id': 'doc_kafka',
        'text': (
            "Apache Kafka is a distributed event streaming platform. It excels at high-throughput, "
            "fault-tolerant, publish-subscribe messaging. Kafka is often used together with Apache "
            "Spark for real-time stream processing pipelines. It was originally developed at LinkedIn "
            "and later became an Apache project."
        )
    },
    {
        'id': 'doc_nosql',
        'text': (
            "NoSQL databases are designed for large-scale, distributed data storage. Apache Cassandra "
            "is a wide-column store optimized for high availability (AP in the CAP theorem). MongoDB "
            "is a document database that prioritizes consistency (CP in the CAP theorem). Neo4j is a "
            "graph database that uses the Cypher query language for pattern matching on graph data."
        )
    },
    {
        'id': 'doc_aws',
        'text': (
            "Amazon Web Services (AWS) provides cloud infrastructure for big data workloads. "
            "Amazon S3 is an object storage service widely used as a data lake. AWS EMR "
            "(Elastic MapReduce) is a managed cluster service that runs Apache Hadoop and "
            "Apache Spark, making it easy to process large datasets without managing hardware."
        )
    },
    {
        'id': 'doc_kg',
        'text': (
            "Knowledge Graphs represent data as nodes and edges where statements are triples of "
            "subject-predicate-object. RDF (Resource Description Framework) is the standard model. "
            "SPARQL is the query language for RDF graphs. Property Graphs, used in Neo4j, have no "
            "formal semantics but support efficient graph traversal. Ontologies define the classes "
            "and properties in a knowledge graph."
        )
    },
]

corpus_texts = [d['text'] for d in documents]
corpus_ids   = [d['id']   for d in documents]

# ── Build TF-IDF index ──────────────────────────────────────────────────────
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(corpus_texts)

print(f'✅ TF-IDF index built: {len(corpus_texts)} documents, {tfidf_matrix.shape[1]} features')

In [ ]:
def vector_retrieve(question: str, top_k: int = 2) -> str:
    """
    Retrieve top-k most relevant documents using TF-IDF cosine similarity.
    """
    q_vec = vectorizer.transform([question])
    scores = cosine_similarity(q_vec, tfidf_matrix).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    retrieved = []
    for idx in top_indices:
        if scores[idx] > 0:
            retrieved.append(f"[{corpus_ids[idx]}, score={scores[idx]:.3f}]\n{corpus_texts[idx]}")
    
    return '\n\n'.join(retrieved) if retrieved else "[No relevant documents found]"


# Test
question = "What processing types does Spark support and what does it use?"
print(f'Question: {question}')
print('\nVector-Retrieved Context:')
print(vector_retrieve(question))

---
## Part 5 – The RAG Pipeline

Now we combine both retrieval methods into a **RAG prompt**.

In [ ]:
def build_rag_prompt(
    question: str,
    use_kg: bool = True,
    use_vector: bool = True
) -> str:
    """
    Build a RAG prompt combining KG facts and/or vector-retrieved documents.
    """
    context_parts = []
    
    if use_kg:
        kg_context = kg_retrieve(question)
        if '[No relevant' not in kg_context:
            context_parts.append(f"### Knowledge Graph Facts:\n{kg_context}")
    
    if use_vector:
        vec_context = vector_retrieve(question, top_k=2)
        if '[No relevant' not in vec_context:
            context_parts.append(f"### Retrieved Documents:\n{vec_context}")
    
    context = '\n\n'.join(context_parts) if context_parts else "[No context retrieved]"
    
    prompt = f"""You are a Big Data expert assistant. Use the provided context to answer the question accurately.

## Context
{context}

## Question
{question}

## Answer
Based on the provided context:"""
    return prompt


# Build the prompt for our example question
question = "What are the processing types of Apache Spark and what storage systems does it relate to?"
prompt = build_rag_prompt(question)
print(prompt)

### 5.1 Send the RAG Prompt to Gemini (optional)

If you have a Google Gemini API key, you can send the prompt to Gemini and get a grounded answer.
You can create a free API key at https://aistudio.google.com/app/apikey

In [ ]:
import os

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")  # set your key here or in env

def call_llm(prompt: str, api_key: str = GEMINI_API_KEY) -> str:
    """Call Gemini via the Google Gen AI SDK."""
    if not api_key:
        return ("[API key not set. Set GEMINI_API_KEY to call the LLM. "
                "The prompt above shows what would be sent.]")
    try:
        from google import genai
        from google.genai import types
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(max_output_tokens=512),
        )
        return response.text
    except Exception as e:
        return f"[LLM call failed: {e}]"


answer = call_llm(prompt)
print("=" * 60)
print("LLM ANSWER")
print("=" * 60)
print(answer)

---
## Part 6 – Compare: KG-RAG vs. Vector-RAG vs. No-RAG

We now run the same set of questions through three configurations and compare the prompts.

In [ ]:
test_questions = [
    "What is the CAP theorem property of Apache Cassandra?",
    "Does Spark support streaming?",
    "What query language does Neo4j use?",
    "What big data services does AWS offer?",
]

print("=" * 70)
print("COMPARISON: KG-RAG  vs  Vector-RAG  vs  No Context")
print("=" * 70)

for q in test_questions:
    print(f"\n❓ Question: {q}")
    print("─" * 60)
    
    kg_ctx = kg_retrieve(q)
    vec_ctx = vector_retrieve(q, top_k=1)
    
    print("  [KG Facts]:")
    for line in kg_ctx.split('\n')[:5]:  # show first 5 facts
        print(f"    {line}")
    
    print("  [Vector Doc snippet]:")
    first_line = vec_ctx.split('\n')[0] if vec_ctx else ''
    print(f"    {first_line}")
    
    # Call LLM for each configuration (only if API key is set)
    # kg_answer = call_llm(build_rag_prompt(q, use_kg=True,  use_vector=False))
    # vec_answer = call_llm(build_rag_prompt(q, use_kg=False, use_vector=True))
    # combined_answer = call_llm(build_rag_prompt(q, use_kg=True, use_vector=True))

---
## Part 7 – Analysis & Discussion

### 7.1 Retrieval Precision Comparison

In [ ]:
# For each question, how many facts did KG vs. vector retrieval return?

results = []
for q in test_questions:
    kg_facts = [f for f in kg_retrieve(q).split('\n') if f.strip() and '[No' not in f]
    vec_docs  = [d for d in vector_retrieve(q, top_k=3).split('\n\n') if d.strip() and '[No' not in d]
    results.append({
        'question': q[:40] + '...',
        'kg_facts': len(kg_facts),
        'vec_docs': len(vec_docs)
    })

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(results))
width = 0.35
ax.bar(x - width/2, [r['kg_facts'] for r in results],  width, label='KG Facts',      color='#4C9BE8')
ax.bar(x + width/2, [r['vec_docs']  for r in results],  width, label='Vector Docs',   color='#F4A261')
ax.set_xticks(x)
ax.set_xticklabels([r['question'] for r in results], rotation=15, ha='right', fontsize=8)
ax.set_ylabel('Number of retrieved items')
ax.set_title('KG Retrieval vs. Vector Retrieval – Items Retrieved per Question')
ax.legend()
plt.tight_layout()
plt.show()

### 7.2 Discussion Questions

Think about the following and write your answers in the cells below:

1. **Precision vs. Recall**: KG retrieval returns specific, structured facts. Vector retrieval returns full document passages. In which scenarios is each approach preferable?

2. **Structured vs. Unstructured knowledge**: What types of questions is each approach better suited for? Give concrete examples.

3. **Hallucination risk**: How does providing KG-grounded facts change the risk of the LLM making up incorrect information?

4. **Scalability**: How would each approach scale to millions of entities or millions of documents?

5. **Knowledge graph construction**: Where does the KG come from in a real system? What are the challenges?

**Your answer to Q1:**

*(double-click to edit)*

**Your answer to Q2:**

*(double-click to edit)*

**Your answer to Q3:**

*(double-click to edit)*

---
## Part 8 – Extension: Graph Traversal for Multi-Hop Reasoning

One key advantage of KGs is **multi-hop reasoning**: following chains of relationships to answer complex questions.

Example: *"What framework uses the storage system with a default replication factor of 3?"*  
→ HDFS has replicationFactor 3 → Hadoop uses HDFS

In [ ]:
# ── Multi-hop SPARQL: 2-hop reasoning ──────────────────────────────────────
q_multihop = """
PREFIX bd: <http://bigdata.example.org/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?framework ?storage ?replication WHERE {
    ?framework bd:uses ?storage .
    ?storage bd:replicationFactor ?replication .
    FILTER(?replication >= 3)
}
"""
run_sparql(q_multihop, "Multi-hop: Which frameworks use storage with replication ≥ 3?")

In [ ]:
# ── SPARQL PATH query: transitive relationships ─────────────────────────────
# Which technologies are transitively related to Spark (via any relationship)?
q_path = """
PREFIX bd: <http://bigdata.example.org/>

SELECT DISTINCT ?related WHERE {
    bd:Spark (bd:uses|bd:usedWith|bd:fasterThan|bd:developedBy)+ ?related .
}
"""
run_sparql(q_path, "Path query: Technologies reachable from Spark")

### ✏️ Extension Exercise: Enhance the Knowledge Graph

Add at least **5 new triples** to the graph to represent facts you know about Big Data technologies.  
Then write a SPARQL query that can only be answered using your new triples.

In [ ]:
# Add your own triples here:
# Example: add(BD.Flink, RDF.type, BD.Framework)

# add(BD.????, ???, ???)

print(f'Graph now has {len(g)} triples')

# Then write a SPARQL query that uses your new triples:
q_extension = """
PREFIX bd: <http://bigdata.example.org/>

SELECT ?s ?p ?o WHERE {
    # TODO: your query here
}
"""
# run_sparql(q_extension, "My extension query")

---
## Summary

In this exercise you built a complete **Graph-Augmented RAG pipeline**:

| Step | What you did |
|------|-------------|
| **KG Construction** | Modeled Big Data technologies as RDF triples with classes, instances, and properties |
| **SPARQL Querying** | Queried the graph with pattern matching, filters, and path expressions |
| **Graph Retrieval** | Linked entities in questions to graph nodes and fetched relevant subgraphs |
| **Vector Retrieval** | Built a TF-IDF index and retrieved relevant documents by cosine similarity |
| **RAG Prompt** | Combined structured KG facts + unstructured text into a grounded LLM prompt |
| **Comparison** | Evaluated the strengths of each retrieval approach |

### Key Takeaways
- **Knowledge Graphs** provide **precise, structured, verifiable** facts — ideal for factual Q&A
- **Vector retrieval** captures **semantic similarity** across unstructured text — ideal for broader context
- **Combining both** (hybrid RAG) gives the best of both worlds
- SPARQL's **multi-hop reasoning** enables answering complex questions that require following chains of relationships — impossible with vector retrieval alone